In [ ]:
# Ensure all necessary libraries are installed
!pip install yt-dlp ultralytics supervision opencv-python numpy

In [ ]:
import subprocess
import os
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO
from IPython.display import Video

# --- Download the video using yt-dlp ---
youtube_url = "https://www.youtube.com/watch?v=WxgtahHmhiw"
input_video_filename = "traffic_video.mp4"
# Temporary output from OpenCV, will be converted
temp_output_video_filename = "traffic_output_temp.mp4"
final_output_video_filename = "traffic_output.mp4"

command = [
    "yt-dlp",
    "-f", "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]", # Prefer MP4 format
    "--output", input_video_filename,
    youtube_url
]

print(f"Downloading video from {youtube_url} to {input_video_filename}...")
try:
    subprocess.run(command, capture_output=True, text=True, check=True)
    print("Download successful!")
    if os.path.exists(input_video_filename):
        print(f"Video saved as: {os.path.abspath(input_video_filename)}")
    else:
        print("Error: Video file not found after download.")
except subprocess.CalledProcessError as e:
    print(f"Error during download: {e}")
    print("Standard Output:", e.stdout)
    print("Standard Error:", e.stderr)
except FileNotFoundError:
    print("Error: yt-dlp command not found. Ensure it's installed and in your PATH.")

# --- Load the YOLOv8 model ---
print("\nLoading YOLOv8 model...")
model = YOLO('yolov8n.pt')
print("YOLOv8 model loaded successfully.")

# --- Set up video capture and writer ---
video = cv2.VideoCapture(input_video_filename)

if not video.isOpened():
    print(f"Error: Could not open input video file {input_video_filename}")
    exit()

W = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = int(video.get(cv2.CAP_PROP_FPS))
TOTAL_FRAMES = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"\nVideo details: Width={W}, Height={H}, FPS={FPS}, Total Frames={TOTAL_FRAMES}")

# Use 'mp4v' or 'XVID' for OpenCV writer, then re-encode with ffmpeg for compatibility
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(temp_output_video_filename, fourcc, FPS, (W, H))

if not out.isOpened():
    print(f"Error: Could not open output video file {temp_output_video_filename}")
    video.release()
    exit()
print(f"Output video writer initialized for {temp_output_video_filename}")

# --- Initialize annotators and LineZone for counting ---
bounding_box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_thickness=2, text_scale=1)

# Define a horizontal virtual counting line near the middle of the frame
start_point = sv.Point(0, H // 2)
end_point = sv.Point(W, H // 2)
line_zone = sv.LineZone(start=start_point, end=end_point)

print("Annotators and LineZone initialized.")

# --- Process each frame for tracking, annotation, and counting ---
frame_idx = 0
print("\nRunning YOLOv8 tracking, annotating, and counting on the video. This may take a while...")
while True:
    ret, frame = video.read()

    if not ret:
        break

    # Run YOLOv8 model on the current frame with tracking enabled
    # verbose=False to suppress per-frame output from model.track
    results = model.track(frame, persist=True, verbose=False)[0]

    # Create a sv.Detections object from the YOLOv8 results
    detections = sv.Detections(
        xyxy=results.boxes.xyxy.cpu().numpy(),
        tracker_id=results.boxes.id.cpu().numpy().astype(int),
        confidence=results.boxes.conf.cpu().numpy(),
        class_id=results.boxes.cls.cpu().numpy().astype(int)
    )

    # Update the line_zone with the current frame's detections
    line_zone.trigger(detections=detections)

    # Prepare the labels for the LabelAnnotator
    labels = [
        f"#{tracker_id} {model.names[class_id]} {confidence:.2f}"
        for xyxy, tracker_id, confidence, class_id
        in zip(detections.xyxy, detections.tracker_id, detections.confidence, detections.class_id)
    ]

    # Annotate the frame with bounding boxes and labels
    annotated_frame = bounding_box_annotator.annotate(scene=frame.copy(), detections=detections)
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)

    # Draw the counting line manually
    cv2.line(annotated_frame, (int(start_point.x), int(start_point.y)),
             (int(end_point.x), int(end_point.y)), (0, 255, 0), 2) # Green line

    # Display counts manually
    text_in = f"In: {line_zone.in_count}"
    text_out = f"Out: {line_zone.out_count}"

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    font_thickness = 2
    text_color = (255, 255, 255) # White color
    text_background_color = (0, 0, 0) # Black background

    (text_in_width, text_in_height), _ = cv2.getTextSize(text_in, font, font_scale, font_thickness)
    cv2.rectangle(annotated_frame, (10, 10), (10 + text_in_width + 10, 10 + text_in_height + 10), text_background_color, -1)
    cv2.putText(annotated_frame, text_in, (20, 10 + text_in_height + 5), font, font_scale, text_color, font_thickness, cv2.LINE_AA)

    (text_out_width, text_out_height), _ = cv2.getTextSize(text_out, font, font_scale, font_thickness)
    cv2.rectangle(annotated_frame, (10, 10 + text_in_height + 20), (10 + text_out_width + 10, 10 + text_in_height + 20 + text_out_height + 10), text_background_color, -1)
    cv2.putText(annotated_frame, text_out, (20, 10 + text_in_height + 20 + text_out_height + 5), font, font_scale, text_color, font_thickness, cv2.LINE_AA)

    # Write the annotated frame to the output video file
    out.write(annotated_frame)

    frame_idx += 1
    if frame_idx % (FPS * 5) == 0: # Print progress every 5 seconds of video
        print(f"Visualizing frame {frame_idx}/{TOTAL_FRAMES}...")

# --- Release resources ---
video.release()
out.release()
print(f"\nFinished processing frames. Temporary video saved to {temp_output_video_filename}")

# --- Re-encode video with ffmpeg for wider compatibility ---
print(f"\nRe-encoding {temp_output_video_filename} to {final_output_video_filename} with ffmpeg...")
try:
    # Using libx264 codec for H.264 video, which is widely supported
    ffmpeg_command = [
        "ffmpeg",
        "-y", # Overwrite output files without asking
        "-i", temp_output_video_filename,
        "-c:v", "libx264",
        "-preset", "medium",
        "-crf", "23",
        "-c:a", "aac",
        "-b:a", "128k",
        final_output_video_filename
    ]
    subprocess.run(ffmpeg_command, capture_output=True, text=True, check=True)
    print("Video re-encoding successful!")
    # Remove the temporary file
    if os.path.exists(temp_output_video_filename):
        os.remove(temp_output_video_filename)
        print(f"Removed temporary file: {temp_output_video_filename}")
except subprocess.CalledProcessError as e:
    print(f"Error during ffmpeg re-encoding: {e}")
    print("Standard Output:", e.stdout)
    print("Standard Error:", e.stderr)
except FileNotFoundError:
    print("Error: ffmpeg command not found. Ensure it's installed and in your PATH.")

# --- Display the processed video ---
print(f"\nDisplaying processed video: {final_output_video_filename}")
Video(final_output_video_filename, embed=True, html_attributes='controls loop')
